# PlainCloak Python Quickstart

Text-safe, authenticated public-key encryption for chat applications.

Each encrypted message is a single pasteable line:
```
PLAINCLOAK:v1:BR:4dHRrngWcgate3V2PFZwBFZFXfOSeE8w...
```

No server, no key exchange protocol, no account required.

---

## Contents
1. [Installation](#1-installation)
2. [Key generation](#2-key-generation)
3. [Encrypt and decrypt](#3-encrypt-and-decrypt)
4. [Direct suite - RSA-only](#4-direct-suite---rsa-only)
5. [Decrypt outcomes](#5-decrypt-outcomes)
6. [Inspect a wire message](#6-inspect-a-wire-message)
7. [PEM serialization](#7-pem-serialization)
8. [Multiple private keys](#8-multiple-private-keys)
9. [Keystore](#9-keystore)
10. [QR transport](#10-qr-transport)

## 1. Installation

```bash
pip install plaincloak           # base install
pip install plaincloak[keystore] # adds Argon2id KDF for the encrypted keystore CLI
pip install plaincloak[qr]       # adds QR encode/decode (optional, Linux needs libzbar0)
```

Requires Python 3.10+.

In [ ]:
%pip install plaincloak[keystore]
%pip install plaincloak[qr]

import plaincloak
from plaincloak import (
    DecryptResult,
    EnvelopeInfo,
    KeyPair,
    Outcome,
    Suite,
    decrypt,
    encrypt,
    generate_keypair,
    key_hash,
    load_private_key_pem,
    load_public_key_pem,
    parse_envelope,
)

print("plaincloak imported OK")

## 2. Key generation

`generate_keypair` returns a frozen `KeyPair(private_key, public_key, key_hash)`.  
The `key_hash` is a SHA-256 fingerprint of the public key - used to identify senders and recipients.

Use 2048-bit keys for development; prefer 4096 in production.

In [ ]:
# Generate keypairs for two parties
alice = generate_keypair(bits=2048)
bob   = generate_keypair(bits=2048)

print(f"Alice key hash: {alice.key_hash}")
print(f"Bob   key hash: {bob.key_hash}")

## 3. Encrypt and decrypt

The default suite is `RSA_OAEP_AES256GCM_SHA256` (hybrid) - a random AES-256-GCM key encrypts the plaintext, then RSA-OAEP wraps that key. No plaintext size cap.

In [ ]:
# Alice encrypts a message for Bob, signing it with her private key
wire = encrypt(
    "meet at the usual place",
    recipient_public_key=bob.public_key,
    sender_private_key=alice.private_key,
)

print(wire)

In [ ]:
# Bob decrypts, providing Alice's public key to verify the signature
result = decrypt(
    wire,
    own_private_keys=[bob.private_key],
    trusted_senders={alice.key_hash: alice.public_key},
)

print(f"Outcome:   {result.outcome}")
print(f"Plaintext: {result.plaintext}")
print(f"Sender:    {result.sender_key_hash}")
print(f"Suite:     {result.suite}")

`Outcome.VERIFIED` means the signature matched a trusted sender and decryption succeeded.

All five possible outcomes are covered in [section 5](#5-decrypt-outcomes).

## 4. Direct suite - RSA-only

`RSA_OAEP_SHA256` encrypts the plaintext directly with RSA-OAEP, skipping the AES layer. Simpler, but caps plaintext at `modulus - 66` bytes (~190 bytes for 2048-bit keys, ~446 bytes for 4096-bit keys). Exceeding the cap raises `PlaintextTooLargeError`.

In [ ]:
from plaincloak import PlaintextTooLargeError

wire_direct = encrypt(
    "short message",
    recipient_public_key=bob.public_key,
    sender_private_key=alice.private_key,
    suite=Suite.RSA_OAEP_SHA256,
)

result_direct = decrypt(
    wire_direct,
    own_private_keys=[bob.private_key],
    trusted_senders={alice.key_hash: alice.public_key},
)

print(f"Outcome:   {result_direct.outcome}")
print(f"Plaintext: {result_direct.plaintext}")
print(f"Suite:     {result_direct.suite}")

# Exceeding the cap raises PlaintextTooLargeError
try:
    encrypt(
        "A" * 500,
        recipient_public_key=bob.public_key,
        sender_private_key=alice.private_key,
        suite=Suite.RSA_OAEP_SHA256,
    )
except PlaintextTooLargeError as e:
    print(f"\nPlaintextTooLargeError: {e}")

## 5. Decrypt outcomes

Cryptographic outcomes are **never raised as exceptions** - they are returned in `result.outcome`. Only structural failures (malformed wire, invalid key) raise exceptions.

| Outcome | Meaning | `plaintext` present? |
|---|---|---|
| `VERIFIED` | Signature valid, sender trusted | Yes |
| `UNKNOWN_SENDER` | Decrypted OK but sender not in `trusted_senders` | Yes |
| `SIGNATURE_INVALID` | Decrypted OK but signature verification failed | Yes |
| `WRONG_RECIPIENT` | No matching private key | No |
| `DECRYPTION_FAILED` | Matching key found but decryption failed | No |

In [ ]:
# Outcome: VERIFIED
r = decrypt(
    wire,
    own_private_keys=[bob.private_key],
    trusted_senders={alice.key_hash: alice.public_key},
)
assert r.outcome is Outcome.VERIFIED
print(f"VERIFIED -> plaintext: {r.plaintext!r}")

In [ ]:
# Outcome: UNKNOWN_SENDER
# trusted_senders is empty - Alice's key is not listed
r = decrypt(
    wire,
    own_private_keys=[bob.private_key],
    trusted_senders={},
)
assert r.outcome is Outcome.UNKNOWN_SENDER
print(f"UNKNOWN_SENDER -> plaintext: {r.plaintext!r}")

In [ ]:
# Outcome: SIGNATURE_INVALID
# Pass a wrong public key under Alice's key hash - the signature won't verify
charlie = generate_keypair(bits=2048)
r = decrypt(
    wire,
    own_private_keys=[bob.private_key],
    trusted_senders={alice.key_hash: charlie.public_key},  # wrong key
)
assert r.outcome is Outcome.SIGNATURE_INVALID
print(f"SIGNATURE_INVALID -> plaintext: {r.plaintext!r}")

In [ ]:
# Outcome: WRONG_RECIPIENT
# Charlie's private key doesn't match the envelope encrypted for Bob
r = decrypt(
    wire,
    own_private_keys=[charlie.private_key],
    trusted_senders={alice.key_hash: alice.public_key},
)
assert r.outcome is Outcome.WRONG_RECIPIENT
print(f"WRONG_RECIPIENT -> plaintext: {r.plaintext!r}")

## 6. Inspect a wire message

`parse_envelope` extracts metadata from a wire message without needing any keys. Useful for routing, logging, or UI display.

In [ ]:
info = parse_envelope(wire)

print(f"Suite:              {info.suite}")
print(f"Message ID:         {info.message_id}")
print(f"Timestamp (ms):     {info.timestamp_ms}")
print(f"Sender key hash:    {info.sender_key_hash}")
print(f"Recipient key hash: {info.recipient_key_hash}")
print(f"Payload length:     {info.payload_len} bytes")
print(f"Body length:        {info.body_len} bytes")

## 7. PEM serialization

Keys are standard `cryptography` library objects. Export them as PEM to persist or share.

In [ ]:
from cryptography.hazmat.primitives import serialization

# Export Alice's public key (SPKI PEM) - safe to share
public_pem: bytes = alice.public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)
print(public_pem.decode())

In [ ]:
# Export Alice's private key (PKCS#8 PEM) - keep secret
private_pem: bytes = alice.private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
print(private_pem.decode()[:300], "...")

In [ ]:
# Load them back with the plaincloak helpers
loaded_pub  = load_public_key_pem(public_pem)
loaded_priv = load_private_key_pem(private_pem)

# key_hash is deterministic from the public key
assert key_hash(loaded_pub) == alice.key_hash
print(f"Loaded key hash matches: {key_hash(loaded_pub) == alice.key_hash}")

In [ ]:
# Round-trip with loaded keys
wire2 = encrypt(
    "keys loaded from PEM",
    recipient_public_key=bob.public_key,
    sender_private_key=loaded_priv,
)

result2 = decrypt(
    wire2,
    own_private_keys=[bob.private_key],
    trusted_senders={key_hash(loaded_pub): loaded_pub},
)

print(f"Outcome:   {result2.outcome}")
print(f"Plaintext: {result2.plaintext}")

## 8. Multiple private keys

Pass a list to `own_private_keys`. `decrypt` tries each key against the recipient key hash in the wire message and stops at the first match. Useful when Bob has rotated keys and may receive messages encrypted to either key.

In [ ]:
# Bob generates a second keypair (e.g. after a key rotation)
bob_new = generate_keypair(bits=2048)

# Old message encrypted to bob's original key
wire_old = encrypt(
    "encrypted before key rotation",
    recipient_public_key=bob.public_key,
    sender_private_key=alice.private_key,
)

# New message encrypted to bob's new key
wire_new = encrypt(
    "encrypted after key rotation",
    recipient_public_key=bob_new.public_key,
    sender_private_key=alice.private_key,
)

# Decrypt both with both keys in the list
trusted = {alice.key_hash: alice.public_key}

for label, w in [("old", wire_old), ("new", wire_new)]:
    r = decrypt(
        w,
        own_private_keys=[bob.private_key, bob_new.private_key],
        trusted_senders=trusted,
    )
    print(f"{label} wire -> {r.outcome}: {r.plaintext!r}")

## 9. Keystore

The keystore is an encrypted JSON file that holds your own keypairs (private keys locked with a passphrase) and trusted contacts (public keys only). It is the same storage backend the CLI uses.

`Keystore.init` selects Argon2id automatically when `plaincloak[keystore]` is installed; falls back to PBKDF2-SHA256 on the base install.

In [ ]:
import tempfile
from pathlib import Path

from plaincloak import (
    ContactEntry,
    Keystore,
    OwnKeyEntry,
    decrypt,
    encrypt,
    generate_keypair,
    key_hash,
)

# Use a temp directory so this notebook is self-contained
_tmp = tempfile.mkdtemp()
ks_path = Path(_tmp) / "keystore.json"
passphrase = b"correct horse battery staple"

# --- Create a new keystore ---
store = Keystore.init(ks_path, passphrase)
print(f"Keystore created at: {ks_path}")

In [ ]:
# --- Generate a keypair and store it ---
my_keypair = generate_keypair(bits=2048)
entry: OwnKeyEntry = store.add_my_key("Personal", my_keypair.private_key)
store.save()

print(f"Stored key  label:      {entry.label}")
print(f"            key_hash:   {entry.key_hash}")
print(f"            created_at: {entry.created_at}")

In [ ]:
# --- Add a contact (public key only, no passphrase needed) ---
from cryptography.hazmat.primitives import serialization

contact_keypair = generate_keypair(bits=2048)  # simulates a friend's key
contact_pem = contact_keypair.public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)

contact: ContactEntry = store.add_contact("Alice", contact_pem, notes="colleague")
store.save()

print(f"Added contact alias:    {contact.alias}")
print(f"              key_hash: {contact.key_hash}")
print(f"              notes:    {contact.notes}")

In [ ]:
# --- List keys and contacts ---
store2 = Keystore.load(ks_path)  # no passphrase needed for public data

print("Own keys:")
for k in store2.list_my_keys():
    print(f"  {k.label:12}  {k.key_hash[:16]}...  expires={k.expires_at}")

print("\nContacts:")
for c in store2.list_contacts():
    print(f"  {c.alias:12}  {c.key_hash[:16]}...  notes={c.notes!r}")

In [ ]:
# --- Encrypt to a contact using stored keys ---
store3 = Keystore.load(ks_path, passphrase)

# entry.public_key is already a loaded RSAPublicKey - no PEM parsing needed
recipient_entry = store3.lookup_by_alias_or_hash("Alice")
sender_key = store3.decrypt_private_key("Personal")

wire = encrypt(
    "hello from the keystore",
    recipient_public_key=recipient_entry.public_key,
    sender_private_key=sender_key,
)
print(wire[:80], "...")

In [ ]:
# --- Decrypt using all stored keys and contacts ---
# Simulate Alice sending a message back to us
wire_from_alice = encrypt(
    "hello back from Alice",
    recipient_public_key=my_keypair.public_key,
    sender_private_key=contact_keypair.private_key,
)

# Build own_private_keys and trusted_senders from the keystore in one step
own_private_keys = [
    store3.decrypt_private_key(k.key_hash)
    for k in store3.list_my_keys()
]
trusted_senders = {
    c.key_hash: c.public_key
    for c in store3.list_contacts()
}

result = decrypt(
    wire_from_alice,
    own_private_keys=own_private_keys,
    trusted_senders=trusted_senders,
)

print(f"Outcome:   {result.outcome}")
print(f"Plaintext: {result.plaintext}")

## 10. QR transport

With the `[qr]` extra installed, a wire can be encoded into a single QR image (handy for air-gapped or screen-to-camera transfer).

`encode_qr` returns a Pillow image, `decode_qr` reads one back from a saved file, and `max_qr_wire_bytes` reports the single-QR capacity per error-correction level.
An oversized wire raises `MessageTooLargeForQRError`.

Install with `pip install plaincloak[qr]` (on Linux, also `apt-get install libzbar0`).

In [ ]:
from IPython.display import display

from plaincloak import (
    MessageTooLargeForQRError,
    decode_qr,
    encode_qr,
    max_qr_wire_bytes,
)

ec_level = 'M'  # one of ['L', 'M', 'Q', 'H']
max_bytes = max_qr_wire_bytes(ec_level)

print(f"Single-QR capacity at EC level {ec_level}: {max_bytes} bytes")
print(f"This wire is {len(wire)} bytes ({len(wire)/max_bytes*100:.2f}%)")

# Encode the wire into a QR image and save it as a PNG  (wire from section 9)
qr_img = encode_qr(wire, error_correction=ec_level)
qr_path = Path(_tmp) / "message.png"
qr_img.save(qr_path)
display(qr_img)

# Decode it back from the saved file
recovered = decode_qr(qr_path)
print(f"Round-trip matches original wire: {recovered == wire}")
print(f"Recovered: {recovered[:60]}...")

# An oversized wire fails cleanly before touching the QR backend
try:
    encode_qr("x" * (max_bytes + 1))
except MessageTooLargeForQRError as e:
    print(f"MessageTooLargeForQRError: {e}")